# Training Transformer Models for Italian NER

This notebook fine-tunes **UmBERTo** (`Musixmatch/umberto-commoncrawl-cased-v1`) on the NERMuD 2023 dataset for Italian Named Entity Recognition.

The companion notebook `03b_training_xlmr.ipynb` fine-tunes **XLM-RoBERTa** (`FacebookAI/xlm-roberta-base`) under the same conditions so that both models can be compared in `04_evaluation.ipynb`.

**Task** — token classification (BIO tagging) with 7 labels: `O`, `B-PER`, `I-PER`, `B-ORG`, `I-ORG`, `B-LOC`, `I-LOC`

## 1. Imports

The libraries below are used for dataset handling, Transformer tokenisation and modelling, and training:

- **`datasets`** — Hugging Face `Dataset` objects and `set_seed` for reproducibility
- **`transformers`** — `AutoTokenizer`, `AutoModelForTokenClassification`, `TrainingArguments`, `Trainer`, `DataCollatorForTokenClassification`
- **`torch`** — GPU tensor operations and CUDA availability check
- **`src.*`** — project utility modules for data loading, tokenisation/label alignment, model creation, and metrics

In [1]:
import json
import os
import random
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    SEED,
    MODEL_1,
    MAX_LENGTH,
    LEARNING_RATE,
    BATCH_SIZE,
    NUM_EPOCHS,
    WEIGHT_DECAY,
    LABEL_NAMES,
    LABEL2ID,
    ID2LABEL,
    set_seed,
)
from src.data_utils import (
    load_jsonl,
    save_jsonl,
    tokenize_and_align_labels,
    prepare_datasets,
)
from src.metrics import compute_metrics
from src.models_utils import get_model


## 2. Reproducibility

A fixed seed (`SEED = 42`) is passed to `set_seed`, which seeds Python's `random`, NumPy, and PyTorch (including CUDA) simultaneously. This ensures that weight initialisations, data shuffling, and dropout are deterministic across runs.

In [2]:
set_seed()

print("Seed:", SEED)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Seed: 42
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## 3. Configuration

The experiments compare an Italian-specific Transformer with a multilingual one under identical hyperparameters:

| Hyperparameter | Value |
|----------------|-------|
| `MODEL_1` | `Musixmatch/umberto-commoncrawl-cased-v1` |
| `MAX_LENGTH` | 256 tokens |
| `LEARNING_RATE` | 2 × 10⁻⁵ |
| `BATCH_SIZE` | 8 |
| `EPOCHS` | 3 |
| `WEIGHT_DECAY` | 0.01 |

`max_length = 256` was chosen based on the EDA finding that the longest sequence in the corpus is 251 tokens (see `02_eda.ipynb`). Using 256 adds a small buffer for the special `[CLS]` / `[SEP]` tokens inserted by the Transformer tokeniser.

In [3]:
print("Model 1:", MODEL_1)
print("Max length:", MAX_LENGTH)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", NUM_EPOCHS)

Model 1: Musixmatch/umberto-commoncrawl-cased-v1
Max length: 256
Learning rate: 2e-05
Batch size: 8
Epochs: 3


## 4. Load Processed Datasets

The preprocessing notebook produced three processed files:

- `train.jsonl`
- `dev.jsonl`
- `test.jsonl`

The training set is used to fine-tune the models, the development set is used for model selection, and the test set will only be used during the final evaluation.

In [4]:
train_data = load_jsonl("../data/processed/train.jsonl")
dev_data = load_jsonl("../data/processed/dev.jsonl")
test_data = load_jsonl("../data/processed/test.jsonl")

print("Train:", len(train_data))
print("Dev:", len(dev_data))
print("Test:", len(test_data))


Train: 27482
Dev: 4767
Test: 4126


## 5. Convert to Hugging Face Datasets

The processed JSONL data is converted into Hugging Face `Dataset` objects so that it can be efficiently tokenized and used by the Trainer API.

In [5]:
train_dataset = Dataset.from_list(train_data)
dev_dataset = Dataset.from_list(dev_data)
test_dataset = Dataset.from_list(test_data)

train_dataset

Dataset({
    features: ['tokens', 'labels', 'domain'],
    num_rows: 27482
})

## 6. Define BIO Labels

The NER task uses the BIO tagging scheme.

The labels are:

- `O`: outside an entity
- `B-X`: beginning of an entity of type X
- `I-X`: continuation of an entity of type X

The entity types are PER, ORG, and LOC.

In [6]:
label_names = sorted(
    set(
        label
        for sample in train_data
        for label in sample["labels"]
    )
)

label2id = {
    label: i
    for i, label in enumerate(label_names)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

print("Labels:", label_names)
print("Number of labels:", len(label_names))
print("label2id:", label2id)

Labels: ['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']
Number of labels: 7
label2id: {'B-LOC': 0, 'B-ORG': 1, 'B-PER': 2, 'I-LOC': 3, 'I-ORG': 4, 'I-PER': 5, 'O': 6}


## 7. Tokenization and Label Alignment

The processed dataset contains word-level tokens and BIO labels.

However, Transformer tokenizers may split a single word into multiple subword tokens. Therefore, the original word-level labels must be aligned with the resulting subwords.

Special tokens receive the label `-100`, meaning that they are ignored by the loss function.

For words split into multiple subwords, only the first subword receives the original BIO label. The remaining subwords are ignored during training.

In [7]:
# tokenize_and_align_labels is imported from src.data_utils


## 8. Model Preparation

The following function prepares the tokenizer and tokenized datasets for a given pretrained Transformer.

The same preprocessing pipeline is used for both models.

In [8]:
# prepare_datasets is imported from src.data_utils


## 9. Testing with UmBERTo

In [9]:

tokenizer_umberto, train_umberto, dev_umberto, test_umberto = (
    prepare_datasets(MODEL_1)
)

print(tokenizer_umberto)

Map:   0%|          | 0/27482 [00:00<?, ? examples/s]

Map:   0%|          | 0/4767 [00:00<?, ? examples/s]

Map:   0%|          | 0/4126 [00:00<?, ? examples/s]

CamembertTokenizerFast(name_or_path='Musixmatch/umberto-commoncrawl-cased-v1', vocab_size=32005, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>', 'additional_special_tokens': ['<s>NOTUSED', '</s>NOTUSED', '<unk>NOTUSED']}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<s>NOTUSED", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>NOTUSED", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<unk>NOTUSED", rstrip=False, lstrip=False, si

In [10]:
print(train_umberto[0])

{'labels': [-100, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, -100, -100], 'input_ids': [5, 245, 4993, 388, 17954, 56, 13063, 2331, 120, 4028, 339, 10102, 6], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## 10. Inspect token/label alignment

In [11]:
example = train_umberto[0]

tokens = tokenizer_umberto.convert_ids_to_tokens(
    example["input_ids"]
)

labels = [
    label_names[label] if label != -100 else "IGN"
    for label in example["labels"]
]

for token, label in zip(tokens, labels):
    print(f"{token:20} {label}")

<s>                  IGN
▁Il                  B-LOC
▁ministro            B-LOC
▁degli               B-LOC
▁Esteri              B-LOC
▁al                  B-LOC
▁commissario         B-LOC
▁capo                B-LOC
▁della               B-LOC
▁Commissione         I-LOC
▁alle                I-ORG
ata                  IGN
</s>                 IGN


## 11. Check Sequence Lengths

The maximum sequence length is set to 256 Transformer tokens.

This check verifies whether the chosen value causes substantial truncation.

In [12]:
lengths = [
    len(example["input_ids"])
    for example in train_umberto
]

print("Maximum tokenized length:", max(lengths))
print("Mean tokenized length:", np.mean(lengths))
print("95th percentile:", np.percentile(lengths, 95))
print("99th percentile:", np.percentile(lengths, 99))

print(
    "Sequences at maximum length:",
    sum(length >= MAX_LENGTH for length in lengths)
)

Maximum tokenized length: 256
Mean tokenized length: 25.472600247434684
95th percentile: 59.0
99th percentile: 86.0
Sequences at maximum length: 2


## 12. Evaluation Metrics

The models are evaluated using **entity-level** NER metrics computed by [seqeval](https://github.com/chakki-works/seqeval):

- **Precision** — fraction of predicted entities that are correct
- **Recall** — fraction of gold entities that are predicted
- **F1** — harmonic mean of precision and recall

Unlike token-level accuracy, seqeval only counts an entity as correct if the **entire span** and its **type** match the gold annotation. This is the standard metric for NER evaluation.

In [13]:
# compute_metrics is imported from src.metrics


## 13. Data Collator

`DataCollatorForTokenClassification` pads input sequences and label sequences to the same length within each batch and sets label values to `-100` for padding positions. Loss is not computed on positions with label `-100`.

In [14]:
data_collator_umberto = DataCollatorForTokenClassification(
    tokenizer=tokenizer_umberto
)

# Model 1 — UmBERTo

**UmBERTo** (`Musixmatch/umberto-commoncrawl-cased-v1`) is an Italian-specific pretrained Transformer based on the RoBERTa architecture.

- Pre-trained on a large Italian Common Crawl corpus
- Vocabulary: ~32,005 tokens (Italian-optimised, cased BPE)
- Architecture: 12 transformer layers, 768 hidden dimensions, ~125 M parameters

Because UmBERTo was pre-trained exclusively on Italian text, it is expected to produce a more efficient and accurate tokenisation of Italian words compared to a multilingual model, leading to better NER performance.

In [15]:
model_umberto = get_model(
    MODEL_1,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

model_umberto


Some weights of CamembertForTokenClassification were not initialized from the model checkpoint at Musixmatch/umberto-commoncrawl-cased-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CamembertForTokenClassification(
  (roberta): CamembertModel(
    (embeddings): CamembertEmbeddings(
      (word_embeddings): Embedding(32005, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): CamembertEncoder(
      (layer): ModuleList(
        (0-11): 12 x CamembertLayer(
          (attention): CamembertAttention(
            (self): CamembertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): CamembertSelfOutput(
              (dense): Linear(in_features=768, out_features=7

## UmBERTo Training Configuration

The model is fine-tuned for **3 epochs** using the **AdamW** optimiser with the following hyperparameters:

| Hyperparameter | Value |
|----------------|-------|
| Learning rate | 2 × 10⁻⁵ |
| Per-device batch size | 8 |
| Weight decay | 0.01 |
| Epochs | 3 |
| Evaluation strategy | every epoch |
| Best model selection | highest dev F1 |

The relatively small batch size (8) is used because UmBERTo's sequence length of 256 tokens keeps memory usage manageable on a single GPU. No gradient accumulation or mixed precision is needed.

In [16]:
training_args_umberto = TrainingArguments(
    output_dir="../models/umberto-ner",

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,

    weight_decay=WEIGHT_DECAY,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_strategy="epoch",

    report_to="none",

    seed=SEED,
)

In [17]:
trainer_umberto = Trainer(
    model=model_umberto,
    args=training_args_umberto,

    train_dataset=train_umberto,
    eval_dataset=dev_umberto,

    data_collator=data_collator_umberto,

    compute_metrics=compute_metrics,
)

## Fine-tuning UmBERTo

Training runs for 3 epochs (~18 minutes on a single GPU). Development set F1 scores at the end of each epoch:

| Epoch | Dev F1 |
|-------|--------|
| 1     | 0.8602 |
| 2     | 0.8724 |
| 3     | 0.8834 |

The model continues to improve across all three epochs, with the largest gain in epoch 1. The best checkpoint (epoch 3) is saved to `../models/umberto-ner-best` and evaluated on the development set immediately after training.

In [18]:
trainer_umberto.train()

  0%|          | 0/10308 [00:00<?, ?it/s]

{'loss': 0.0825, 'grad_norm': 0.013339927420020103, 'learning_rate': 1.3333333333333333e-05, 'epoch': 1.0}


  0%|          | 0/596 [00:00<?, ?it/s]

{'eval_loss': 0.045469980686903, 'eval_precision': 0.8521722776001505, 'eval_recall': 0.8683403602912994, 'eval_f1': 0.8601803512102516, 'eval_accuracy': 0.9861846051912935, 'eval_runtime': 9.6767, 'eval_samples_per_second': 492.627, 'eval_steps_per_second': 61.591, 'epoch': 1.0}
{'loss': 0.0224, 'grad_norm': 0.04172152653336525, 'learning_rate': 6.666666666666667e-06, 'epoch': 2.0}


  0%|          | 0/596 [00:00<?, ?it/s]

{'eval_loss': 0.03973936662077904, 'eval_precision': 0.8645088445615355, 'eval_recall': 0.8804139517056343, 'eval_f1': 0.8723889099886062, 'eval_accuracy': 0.9878768343674734, 'eval_runtime': 9.4687, 'eval_samples_per_second': 503.449, 'eval_steps_per_second': 62.944, 'epoch': 2.0}
{'loss': 0.0135, 'grad_norm': 0.00787835381925106, 'learning_rate': 0.0, 'epoch': 3.0}


  0%|          | 0/596 [00:00<?, ?it/s]

{'eval_loss': 0.04220026731491089, 'eval_precision': 0.885912507226826, 'eval_recall': 0.8809888846301265, 'eval_f1': 0.8834438358796963, 'eval_accuracy': 0.9888885904357714, 'eval_runtime': 9.4816, 'eval_samples_per_second': 502.761, 'eval_steps_per_second': 62.858, 'epoch': 3.0}
{'train_runtime': 1070.8879, 'train_samples_per_second': 76.988, 'train_steps_per_second': 9.626, 'train_loss': 0.03949037003433885, 'epoch': 3.0}


TrainOutput(global_step=10308, training_loss=0.03949037003433885, metrics={'train_runtime': 1070.8879, 'train_samples_per_second': 76.988, 'train_steps_per_second': 9.626, 'total_flos': 2330074215679848.0, 'train_loss': 0.03949037003433885, 'epoch': 3.0})

### Training Progress

The training loop logs loss and development-set metrics at the end of each epoch. The table above summarises the progression. Steady improvement across epochs suggests the model is learning without overfitting within 3 epochs.

In [19]:
umberto_dev_results = trainer_umberto.evaluate()

umberto_dev_results

  0%|          | 0/596 [00:00<?, ?it/s]

{'eval_loss': 0.04220026731491089,
 'eval_precision': 0.885912507226826,
 'eval_recall': 0.8809888846301265,
 'eval_f1': 0.8834438358796963,
 'eval_accuracy': 0.9888885904357714,
 'eval_runtime': 9.4812,
 'eval_samples_per_second': 502.785,
 'eval_steps_per_second': 62.861,
 'epoch': 3.0}

### Development Set Results

`trainer_umberto.evaluate()` runs inference on the full development split and reports entity-level precision, recall, and F1 computed by seqeval. The best result after epoch 3 is **F1 = 0.8834** on the development set.

In [20]:
print("UmBERTo - Development Set")
print(
    f"Precision: {umberto_dev_results['eval_precision']:.4f}"
)
print(
    f"Recall:    {umberto_dev_results['eval_recall']:.4f}"
)
print(
    f"F1:        {umberto_dev_results['eval_f1']:.4f}"
)
print(
    f"Accuracy:  {umberto_dev_results['eval_accuracy']:.4f}"
)


UmBERTo - Development Set
Precision: 0.8859
Recall:    0.8810
F1:        0.8834
Accuracy:  0.9889


## Save Best Model

The best checkpoint is saved to `../models/umberto-ner-best/`. Both the model weights and the tokenizer are persisted so that the evaluation notebook (`04_evaluation.ipynb`) can load them without retraining.

In [ ]:
trainer_umberto.save_model("../models/umberto-ner-best")
tokenizer_umberto.save_pretrained("../models/umberto-ner-best")

('../models/umberto-ner-best/tokenizer_config.json',
 '../models/umberto-ner-best/special_tokens_map.json',
 '../models/umberto-ner-best/sentencepiece.bpe.model',
 '../models/umberto-ner-best/added_tokens.json',
 '../models/umberto-ner-best/tokenizer.json')

: 